In [ ]:
"""
SCRIPT 04
Menghitung jarak dari setiap desa prioritas
ke FKTP terdekat menggunakan Haversine distance.

Input:
    Output/Desa.csv
    Output/FKTP.csv

Output:
    Output/01_Hasil_Euclidean_Desa_FKTP.csv
    Output/01_Hasil_Euclidean_Desa_FKTP.xlsx
"""


from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder

OUTPUT_DIR = PROJECT_ROOT / "Output"

OUTPUT_DIR.mkdir(
    exist_ok=True
)


DESA_FILE = OUTPUT_DIR / "Desa.csv"

FKTP_FILE = OUTPUT_DIR / "FKTP.csv"

CSV_OUT = OUTPUT_DIR / (
    "01_Hasil_Euclidean_Desa_FKTP.csv"
)

XLSX_OUT = OUTPUT_DIR / (
    "01_Hasil_Euclidean_Desa_FKTP.xlsx"
)


# ============================================================
# KONSTANTA
# ============================================================

R = 6378137.0


# ============================================================
# HAVERSINE
# ============================================================

def haversine(

    lon1,

    lat1,

    lon2,

    lat2

):

    """

    Menghitung jarak Haversine.

    lon1, lat1:
        koordinat satu desa

    lon2, lat2:
        array koordinat seluruh FKTP

    Return:
        jarak dalam meter

    """


    lon1 = np.radians(

        float(lon1)

    )


    lat1 = np.radians(

        float(lat1)

    )


    lon2 = np.radians(

        np.asarray(

            lon2,

            dtype=float

        )

    )


    lat2 = np.radians(

        np.asarray(

            lat2,

            dtype=float

        )

    )


    dlon = lon2 - lon1

    dlat = lat2 - lat1


    a = (

        np.sin(

            dlat / 2

        )

        ** 2

    ) + (

        np.cos(lat1)

        *

        np.cos(lat2)

        *

        np.sin(

            dlon / 2

        )

        ** 2

    )


    return (

        2

        *

        R

        *

        np.arcsin(

            np.sqrt(a)

        )

    )


# ============================================================
# MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # 1. BACA DATA
    # --------------------------------------------------------

    desa = pd.read_csv(

        DESA_FILE

    )


    fktp = pd.read_csv(

        FKTP_FILE

    )


    # --------------------------------------------------------
    # 2. KONVERSI KOORDINAT
    # --------------------------------------------------------

    desa["Longitude"] = pd.to_numeric(

        desa["Longitude"],

        errors="coerce"

    )


    desa["Latitude"] = pd.to_numeric(

        desa["Latitude"],

        errors="coerce"

    )


    fktp["Longitude"] = pd.to_numeric(

        fktp["Longitude"],

        errors="coerce"

    )


    fktp["Latitude"] = pd.to_numeric(

        fktp["Latitude"],

        errors="coerce"

    )


    # --------------------------------------------------------
    # 3. HAPUS KOORDINAT INVALID
    # --------------------------------------------------------

    desa = desa.dropna(

        subset=[

            "Longitude",

            "Latitude"

        ]

    )


    fktp = fktp.dropna(

        subset=[

            "Longitude",

            "Latitude"

        ]

    )


    print(

        f"Jumlah Desa : {len(desa)}"

    )


    print(

        f"Jumlah FKTP : {len(fktp)}"

    )


    # --------------------------------------------------------
    # 4. VALIDASI DATA
    # --------------------------------------------------------

    if len(desa) == 0:

        raise ValueError(

            "Tidak ada desa dengan koordinat valid."

        )


    if len(fktp) == 0:

        raise ValueError(

            "Tidak ada FKTP dengan koordinat valid."

        )


    # --------------------------------------------------------
    # 5. ARRAY KOORDINAT FKTP
    # --------------------------------------------------------

    fktp_lon = (

        fktp["Longitude"]

        .to_numpy(

            dtype=float

        )

    )


    fktp_lat = (

        fktp["Latitude"]

        .to_numpy(

            dtype=float

        )

    )


    # --------------------------------------------------------
    # 6. HITUNG JARAK
    # --------------------------------------------------------

    rows = []


    for d in desa.itertuples(

        index=False

    ):


        jarak = haversine(

            d.Longitude,

            d.Latitude,

            fktp_lon,

            fktp_lat

        )


        i = int(

            np.argmin(

                jarak

            )

        )


        f = fktp.iloc[i]


        rows.append(

            {

                "KDDESA":

                    d.KDDESA,


                "NMDESA2":

                    d.NMDESA2,


                "NMKEC":

                    d.NMKEC,


                "NMDATI2":

                    d.NMDATI2,


                "NMPROP":

                    d.NMPROP,


                "Longitude_Desa":

                    d.Longitude,


                "Latitude_Desa":

                    d.Latitude,


                "kode_ppk":

                    f["kode_ppk"],


                "nmppk":

                    f["nmppk"],


                "Longitude_FKTP":

                    f["Longitude"],


                "Latitude_FKTP":

                    f["Latitude"],


                "Jarak_Meter":

                    round(

                        float(

                            jarak[i]

                        ),

                        2

                    ),


                "Jarak_KM":

                    round(

                        float(

                            jarak[i]

                        )

                        /

                        1000,

                        3

                    )

            }

        )


    # --------------------------------------------------------
    # 7. BUAT DATAFRAME HASIL
    # --------------------------------------------------------

    hasil = pd.DataFrame(

        rows

    )


    hasil = hasil.sort_values(

        [

            "NMPROP",

            "NMDATI2",

            "NMKEC",

            "NMDESA2"

        ]

    )


    hasil = hasil.reset_index(

        drop=True

    )


    # --------------------------------------------------------
    # 8. SIMPAN HASIL
    # --------------------------------------------------------

    hasil.to_csv(

        CSV_OUT,

        index=False,

        encoding="utf-8"

    )


    hasil.to_excel(

        XLSX_OUT,

        index=False

    )


    # --------------------------------------------------------
    # 9. RINGKASAN
    # --------------------------------------------------------

    print()

    print(

        "SELESAI"

    )


    print(

        f"Jumlah Desa : {len(hasil)}"

    )


    print(

        f"Minimum     : "

        f"{hasil['Jarak_KM'].min():.2f} km"

    )


    print(

        f"Rata-rata   : "

        f"{hasil['Jarak_KM'].mean():.2f} km"

    )


    print(

        f"Median      : "

        f"{hasil['Jarak_KM'].median():.2f} km"

    )


    print(

        f"Maksimum    : "

        f"{hasil['Jarak_KM'].max():.2f} km"

    )


    print()

    print(

        "CSV tersimpan di:"

    )


    print(

        CSV_OUT

    )


    print()

    print(

        "Excel tersimpan di:"

    )


    print(

        XLSX_OUT

    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()